In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import kruskal

# Load cleaned data
df = pd.read_csv("cleaned_bearing_data.csv", parse_dates=["subscription_start", "timestamp_of_fault"])

# Compute operational days
df['operational_days'] = (df['timestamp_of_fault'] - df['subscription_start']).dt.days
df = df[df['operational_days'].notna()]


In [2]:
df['context_key'] = (
    df['industry_type'].astype(str) + "|" +
    df['machine_type'].astype(str) + "|" +
    df['rpm_min'].astype(str)
)


In [4]:
results = []

for key, group in df.groupby("context_key"):
    make_counts = group['bearing_make'].value_counts()
    valid_makes = make_counts[make_counts >= 10].index.tolist()

    if len(valid_makes) < 2:
        continue  # Need at least 2 makes for comparison

    filtered_group = group[group['bearing_make'].isin(valid_makes)]

    # Kruskal-Wallis test
    groups = [filtered_group[filtered_group['bearing_make'] == make]['operational_days'] for make in valid_makes]
    stat, p_val = kruskal(*groups)

    summary = filtered_group.groupby("bearing_make")['operational_days'].agg(['mean', 'count']).reset_index()
    summary['context'] = key
    summary['p_value'] = p_val
    results.append(summary)

result_df = pd.concat(results, ignore_index=True)
result_df = result_df.sort_values(by=["context", "mean"], ascending=[True, False])
result_df.to_csv("outputs/q6/make_life_comparison_same_context.csv", index=False)


In [5]:
# Filter contexts with significant p-values (p < 0.05)
significant_contexts = result_df[result_df['p_value'] < 0.05]['context'].unique()
sig_df = result_df[result_df['context'].isin(significant_contexts)]

sig_df.to_csv("outputs/q6/significant_make_rankings.csv", index=False)


In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv("cleaned_bearing_data.csv", parse_dates=["subscription_start", "timestamp_of_fault"])

# Compute operational life
df['operational_days'] = (df['timestamp_of_fault'] - df['subscription_start']).dt.days
df = df.dropna(subset=['machine_type', 'rpm_min', 'operational_days'])

# Create context: Machine Type + RPM
df['machine_rpm_context'] = df['machine_type'].astype(str) + " | " + df['rpm_min'].astype(str)

# Group by machine + rpm
summary = (
    df.groupby(['machine_type', 'rpm_min'])
      .agg(avg_life=('operational_days', 'mean'),
           median_life=('operational_days', 'median'),
           count=('operational_days', 'count'))
      .reset_index()
)

# Save for streamlit
summary.to_csv("outputs/q6/machine_rpm_life_summary.csv", index=False)


In [2]:
# Assuming df is already cleaned and operational_days computed
def rpm_range(rpm):
    if rpm < 500:
        return "<500"
    elif rpm < 1000:
        return "500–999"
    elif rpm < 2000:
        return "1000–1999"
    elif rpm < 3000:
        return "2000–2999"
    else:
        return "3000+"

df['rpm_range'] = df['rpm_min'].apply(rpm_range)

summary = df.groupby(['machine_type', 'rpm_range']).agg(
    avg_life=('operational_days', 'mean'),
    median_life=('operational_days', 'median'),
    count=('operational_days', 'count')
).reset_index()

summary.to_csv("outputs/q6/machine_rpm_life_summary.csv", index=False)
